In [1]:
from pathlib import Path
from collections import defaultdict
import pandas as pd
import json

In [2]:
root = Path(r"C:\Users\Ander\Desktop\F1-project\data\raw")

telemetry_files = list(root.rglob("*_tel.json"))

groups = defaultdict(list)

for file in telemetry_files:

    race = file.parts[-4]
    session = file.parts[-3]
    driver = file.parts[-2]

    key = (race, session, driver)

    groups[key].append(file)

In [3]:
processed_root = Path(
    r"C:\Users\Ander\Desktop\F1-project\data\processed"
)

for (race, session, driver), files in groups.items():

    dfs = []

    for file in sorted(files):

        try:

            lap = int(file.stem.split("_")[0])

            with open(file, "r", encoding="utf-8") as f:
                tel = json.load(f)["tel"]

            df = pd.DataFrame(tel)

            # add lap metadata
            df["race"] = race
            df["session"] = session
            df["driver"] = driver
            df["lap"] = lap

            dfs.append(df)

        except Exception as e:

            print(f"ERROR reading {file}")
            print(e)

    # skip empty
    if len(dfs) == 0:
        continue

    # combine laps
    combined = pd.concat(dfs, ignore_index=True)

    # fix string nulls
    combined = combined.replace("None", pd.NA)

    # convert columns to numeric where possible
    for col in combined.columns:

        try:
            combined[col] = pd.to_numeric(
                combined[col],
                errors="ignore"
            )
        except:
            pass

    # output directory
    out_dir = processed_root / race / session
    out_dir.mkdir(parents=True, exist_ok=True)

    # save parquet
    out_file = out_dir / f"{driver}.parquet"

    try:

        combined.to_parquet(
            out_file,
            index=False
        )

        print(f"Saved: {race} | {session} | {driver}")

    except Exception as e:

        print(f"ERROR saving {out_file}")
        print(e)

Saved: Abu Dhabi Grand Prix | Qualifying | ALB
Saved: Abu Dhabi Grand Prix | Qualifying | ALO
Saved: Abu Dhabi Grand Prix | Qualifying | ANT
Saved: Abu Dhabi Grand Prix | Qualifying | BEA
Saved: Abu Dhabi Grand Prix | Qualifying | BOR
Saved: Abu Dhabi Grand Prix | Qualifying | COL
Saved: Abu Dhabi Grand Prix | Qualifying | GAS
Saved: Abu Dhabi Grand Prix | Qualifying | HAD
Saved: Abu Dhabi Grand Prix | Qualifying | HAM
Saved: Abu Dhabi Grand Prix | Qualifying | HUL
Saved: Abu Dhabi Grand Prix | Qualifying | LAW
Saved: Abu Dhabi Grand Prix | Qualifying | LEC
Saved: Abu Dhabi Grand Prix | Qualifying | NOR
Saved: Abu Dhabi Grand Prix | Qualifying | OCO
Saved: Abu Dhabi Grand Prix | Qualifying | PIA
Saved: Abu Dhabi Grand Prix | Qualifying | RUS
Saved: Abu Dhabi Grand Prix | Qualifying | SAI
Saved: Abu Dhabi Grand Prix | Qualifying | STR
Saved: Abu Dhabi Grand Prix | Qualifying | TSU
Saved: Abu Dhabi Grand Prix | Qualifying | VER
Saved: Abu Dhabi Grand Prix | Race | ALB
Saved: Abu Dhabi Gr

In [12]:
import json
from pathlib import Path

import numpy as np
import pandas as pd


# =========================================================
# PATHS
# =========================================================

raw_root = Path(
    r"C:\Users\Ander\Desktop\F1-project\data\raw"
)

processed_root = Path(
    r"C:\Users\Ander\Desktop\F1-project\data\processed"
)

sessions = [
    "Race",
    "Qualifying",
]


# =========================================================
# CLEANER
# =========================================================

def clean_dataframe(df):

    # Replace "None" strings
    df = df.replace("None", np.nan)

    for col in df.columns:

        # Skip numeric
        if pd.api.types.is_numeric_dtype(df[col]):
            continue

        # Handle nested lists/dicts
        sample = df[col].dropna()

        if len(sample) > 0:

            first_val = sample.iloc[0]

            if isinstance(first_val, (list, dict)):

                df[col] = df[col].astype(str)
                continue

        # Try numeric conversion
        numeric = pd.to_numeric(
            df[col],
            errors="coerce"
        )

        # Keep numeric if majority numeric
        if numeric.notna().sum() > 0.7 * len(df):

            df[col] = numeric
            continue

        # Boolean conversion
        non_null = df[col].dropna()

        if len(non_null) > 0:

            unique_vals = non_null.unique()

            if all(
                isinstance(v, (bool, np.bool_))
                for v in unique_vals
            ):

                df[col] = df[col].astype("boolean")
                continue

        # Fallback string
        df[col] = df[col].astype("string")

    return df


# =========================================================
# MAIN
# =========================================================

for race_dir in raw_root.iterdir():

    if not race_dir.is_dir():
        continue

    race_name = race_dir.name

    print(f"\n====================")
    print(race_name)
    print("====================")

    for session in sessions:

        raw_session_dir = race_dir / session

        if not raw_session_dir.exists():
            continue

        processed_session_dir = (
            processed_root
            / race_name
            / session
        )

        print(f"\n--- {session} ---")

        for driver_dir in raw_session_dir.iterdir():

            if not driver_dir.is_dir():
                continue

            driver = driver_dir.name

            lap_json = (
                driver_dir
                / "laptimes.json"
            )

            if not lap_json.exists():
                continue

            try:

                with open(
                    lap_json,
                    "r",
                    encoding="utf-8"
                ) as f:

                    data = json.load(f)

                df = pd.DataFrame(data)

                df = clean_dataframe(df)

                # Metadata
                df["race"] = race_name
                df["session"] = session
                df["driver"] = driver

                output_path = (
                    processed_session_dir
                    / f"{driver}_laps.parquet"
                )

                output_path.parent.mkdir(
                    parents=True,
                    exist_ok=True
                )

                df.to_parquet(
                    output_path,
                    index=False
                )

                print(f"Saved -> {output_path}")

            except Exception as e:

                print(f"FAILED -> {lap_json}")
                print(e)

print("\nDONE.")


Abu Dhabi Grand Prix

--- Race ---
Saved -> C:\Users\Ander\Desktop\F1-project\data\processed\Abu Dhabi Grand Prix\Race\ALB_laps.parquet
Saved -> C:\Users\Ander\Desktop\F1-project\data\processed\Abu Dhabi Grand Prix\Race\ALO_laps.parquet
Saved -> C:\Users\Ander\Desktop\F1-project\data\processed\Abu Dhabi Grand Prix\Race\ANT_laps.parquet
Saved -> C:\Users\Ander\Desktop\F1-project\data\processed\Abu Dhabi Grand Prix\Race\BEA_laps.parquet
Saved -> C:\Users\Ander\Desktop\F1-project\data\processed\Abu Dhabi Grand Prix\Race\BOR_laps.parquet
Saved -> C:\Users\Ander\Desktop\F1-project\data\processed\Abu Dhabi Grand Prix\Race\COL_laps.parquet
Saved -> C:\Users\Ander\Desktop\F1-project\data\processed\Abu Dhabi Grand Prix\Race\GAS_laps.parquet
Saved -> C:\Users\Ander\Desktop\F1-project\data\processed\Abu Dhabi Grand Prix\Race\HAD_laps.parquet
Saved -> C:\Users\Ander\Desktop\F1-project\data\processed\Abu Dhabi Grand Prix\Race\HAM_laps.parquet
Saved -> C:\Users\Ander\Desktop\F1-project\data\process

In [13]:
import json
from pathlib import Path

import numpy as np
import pandas as pd


# =========================================================
# PATHS
# =========================================================

raw_root = Path(
    r"C:\Users\Ander\Desktop\F1-project\data\raw"
)

processed_root = Path(
    r"C:\Users\Ander\Desktop\F1-project\data\processed"
)

sessions = [
    "Race",
    "Qualifying",
]

shared_jsons = [
    "corners",
    "drivers",
    "rcm",
    "session_laptimes",
    "weather",
]


# =========================================================
# CLEANER
# =========================================================

def clean_dataframe(df):

    df = df.replace("None", np.nan)

    for col in df.columns:

        # Skip numeric
        if pd.api.types.is_numeric_dtype(df[col]):
            continue

        # Handle nested lists/dicts
        sample = df[col].dropna()

        if len(sample) > 0:

            first_val = sample.iloc[0]

            if isinstance(first_val, (list, dict)):

                df[col] = df[col].astype(str)
                continue

        # Try numeric conversion
        numeric = pd.to_numeric(
            df[col],
            errors="coerce"
        )

        if numeric.notna().sum() > 0.7 * len(df):

            df[col] = numeric
            continue

        # Boolean conversion
        non_null = df[col].dropna()

        if len(non_null) > 0:

            unique_vals = non_null.unique()

            if all(
                isinstance(v, (bool, np.bool_))
                for v in unique_vals
            ):

                df[col] = df[col].astype("boolean")
                continue

        # Fallback string
        df[col] = df[col].astype("string")

    return df


# =========================================================
# MAIN
# =========================================================

for race_dir in raw_root.iterdir():

    if not race_dir.is_dir():
        continue

    race_name = race_dir.name

    print(f"\n====================")
    print(race_name)
    print("====================")

    for session in sessions:

        raw_session_dir = race_dir / session

        if not raw_session_dir.exists():
            continue

        processed_session_dir = (
            processed_root
            / race_name
            / session
        )

        print(f"\n--- {session} ---")

        for json_name in shared_jsons:

            json_path = (
                raw_session_dir
                / f"{json_name}.json"
            )

            if not json_path.exists():
                continue

            try:

                with open(
                    json_path,
                    "r",
                    encoding="utf-8"
                ) as f:

                    data = json.load(f)

                # Normalize nested json
                df = pd.json_normalize(data)

                df = clean_dataframe(df)

                # Metadata
                df["race"] = race_name
                df["session"] = session

                output_path = (
                    processed_session_dir
                    / f"{json_name}.parquet"
                )

                output_path.parent.mkdir(
                    parents=True,
                    exist_ok=True
                )

                df.to_parquet(
                    output_path,
                    index=False
                )

                print(f"Saved -> {output_path}")

            except Exception as e:

                print(f"FAILED -> {json_path}")
                print(e)

print("\nDONE.")


Abu Dhabi Grand Prix

--- Race ---
Saved -> C:\Users\Ander\Desktop\F1-project\data\processed\Abu Dhabi Grand Prix\Race\corners.parquet
Saved -> C:\Users\Ander\Desktop\F1-project\data\processed\Abu Dhabi Grand Prix\Race\drivers.parquet
Saved -> C:\Users\Ander\Desktop\F1-project\data\processed\Abu Dhabi Grand Prix\Race\rcm.parquet
Saved -> C:\Users\Ander\Desktop\F1-project\data\processed\Abu Dhabi Grand Prix\Race\session_laptimes.parquet
Saved -> C:\Users\Ander\Desktop\F1-project\data\processed\Abu Dhabi Grand Prix\Race\weather.parquet

--- Qualifying ---
Saved -> C:\Users\Ander\Desktop\F1-project\data\processed\Abu Dhabi Grand Prix\Qualifying\corners.parquet
Saved -> C:\Users\Ander\Desktop\F1-project\data\processed\Abu Dhabi Grand Prix\Qualifying\drivers.parquet
Saved -> C:\Users\Ander\Desktop\F1-project\data\processed\Abu Dhabi Grand Prix\Qualifying\rcm.parquet
Saved -> C:\Users\Ander\Desktop\F1-project\data\processed\Abu Dhabi Grand Prix\Qualifying\session_laptimes.parquet
Saved -> C